In [ ]:
# Academic Early Warning System: explore and explain
# READ ONLY. This notebook never fits, publishes, registers or contacts students.
# Requires publication_completed from 01; tables_loaded from 00 alone is not a model publication.
# It follows current.json to immutable uploaded-small or generated-full snapshots.
# Use only your trusted workspace's own MLflow model artifacts.
# Contributions are underlying linear class-score units before calibration,
# not probability changes, percentages, feature significance or causal effects.


In [ ]:
import sys
import types
EMBEDDED_MODULES = {'aews': '"""Academic Early Warning System: synthetic replication starter."""\n', 'aews.runtime': '"""Read-only Fabric runtime and publication checks. No cloud session is created."""\nimport importlib\nfrom importlib.metadata import PackageNotFoundError, version\nfrom pathlib import Path\nimport sys\n\nPURPOSE = "academic_early_warning_current_publication"\nCURRENT_MANIFEST = "Files/model_artifacts/current.json"\nSCHEMAS = ("canvas", "canvas_logs", "sim_history", "sim_support", "silver", "gold")\nTABLES = ("silver.historical_weekly_features", "silver.current_weekly_features",\n          "gold.student_course_predictions", "gold.prediction_explanations",\n          "gold.model_evaluation", "gold.data_quality_summary")\n\n\ndef qualified(config, schema, table=None):\n    if schema not in SCHEMAS and schema != "dbo":\n        raise ValueError("Unexpected schema.")\n    pieces = [config["intended_lakehouse"], schema] + ([table] if table else [])\n    if any(not p or any(c in p for c in "`\\r\\n/\\\\") for p in pieces):\n        raise ValueError("Unsafe catalog name.")\n    return ".".join("`" + piece + "`" for piece in pieces)\n\n\ndef _mlflow_version():\n    try:\n        return version("mlflow")\n    except PackageNotFoundError:\n        module = importlib.import_module("mlflow")\n        value = getattr(module, "__version__", None)\n        if not isinstance(value, str) or not value:\n            raise RuntimeError("MLflow is required for modeling, but its version is unavailable.")\n        return value\n\n\ndef _delta_version(spark):\n    importlib.import_module("delta.tables")\n    spark._jvm.java.lang.Thread.currentThread().getContextClassLoader().loadClass("io.delta.tables.DeltaTable")\n    try:\n        value = version("delta-spark")\n    except PackageNotFoundError:\n        print("Fabric-provided Delta Python and Spark classes are available; exact Delta version metadata is unavailable.")\n        return None\n    if not value.startswith("3.2."):\n        raise RuntimeError(f"Expected Delta 3.2, observed {value}.")\n    return value\n\n\ndef validate_session(spark, config, *, require_model=True):\n    if spark is None:\n        raise RuntimeError("Attach a Lakehouse and use the existing Fabric Spark session.")\n    if sys.version_info[:2] != (3, 11):\n        raise RuntimeError("This starter targets Python 3.11 in Fabric Runtime 1.3.")\n    if not spark.version.startswith("3.5."):\n        raise RuntimeError("This starter targets Spark 3.5 in Fabric Runtime 1.3.")\n    import notebookutils\n    from pyspark.sql import SparkSession\n    if SparkSession.getActiveSession() is None:\n        raise RuntimeError("No active Spark session; this starter never creates one.")\n    context = dict(notebookutils.runtime.context)\n    if not context.get("defaultLakehouseName") or not context.get("defaultLakehouseWorkspaceName"):\n        raise RuntimeError(\n            "The running session has no default Lakehouse. Set the intended Lakehouse as default, "\n            "stop/restart the notebook session, then rerun all cells.")\n    for key, field in (("currentWorkspaceName", "intended_workspace"),\n                       ("defaultLakehouseName", "intended_lakehouse"),\n                       ("defaultLakehouseWorkspaceName", "intended_workspace")):\n        if context.get(key) != config[field]:\n            raise RuntimeError(\n                f"{key} is not the configured target: expected {config[field]!r}, observed {context.get(key)!r}. "\n                "Open the notebook in the intended workspace, right-click its Lakehouse -> Set as default lakehouse, "\n                "then stop/restart the notebook session and rerun all cells.")\n    expected = {"numpy": "1.26.4", "pandas": "2.1.4", "scikit-learn": "1.2.2", "pyarrow": "14.0.2"}\n    observed = {name: version(name) for name in expected}\n    if require_model:\n        expected["mlflow"] = "2.12.2"\n        observed["mlflow"] = _mlflow_version()\n    for name, required in expected.items():\n        if observed[name] != required:\n            raise RuntimeError(f"Expected {name} {required}, observed {observed[name]}; validate compatibility before changing the guard.")\n    if require_model:\n        importlib.import_module("mlflow.sklearn")\n    delta_version = _delta_version(spark)\n    expected_schema = f"{config[\'intended_workspace\']}.{config[\'intended_lakehouse\']}.dbo"\n    namespaces = spark.sql("SHOW SCHEMAS").collect()\n    if not any(str(value).casefold() == expected_schema.casefold() for row in namespaces for value in row):\n        raise RuntimeError("The exact workspace.Lakehouse.dbo namespace is not present.")\n    spark.sql("SHOW TABLES IN " + qualified(config, "dbo")).collect()\n    root = Path("/") / "lakehouse" / "default" / "Files"\n    if not root.is_dir():\n        raise RuntimeError("Default Lakehouse Files mount unavailable.")\n    return root, {**observed, "python": ".".join(map(str, sys.version_info[:3])),\n                  "spark": spark.version, "delta": delta_version,\n                  "delta_capability_verified": True,\n                  "delta_version_source": "package_metadata" if delta_version is not None else "unavailable"}\n\n\ndef validate_manifest(manifest):\n    if manifest.get("purpose") != PURPOSE or manifest.get("publication_state") != "complete":\n        raise RuntimeError("Publication is absent, incomplete or unrelated.")\n    if manifest.get("source_profile") not in ("small", "full") or manifest.get("registered") is not False:\n        raise RuntimeError("Exploration requires a recognized synthetic unregistered publication.")\n    required = ("model_version", "mlflow_run_id", "code_sha256", "source_dataset_version",\n                "train_data_sha256", "publication_archive")\n    if any(not isinstance(manifest.get(key), str) or not manifest[key] for key in required):\n        raise RuntimeError("Publication lacks immutable provenance.")\n    if manifest.get("schemas") != {"silver": "silver", "gold": "gold"}:\n        raise RuntimeError("Publication schemas do not match.")\n    if set(manifest.get("table_paths", {})) != set(TABLES):\n        raise RuntimeError("Publication table snapshots are incomplete.")\n    root = manifest["publication_archive"]\n    if not root.startswith("Files/model_artifacts/") or ".." in root or "\\\\" in root:\n        raise RuntimeError("Unsafe publication archive path.")\n    for table, path in manifest["table_paths"].items():\n        if path != root + "/tables/" + table.replace(".", "/"):\n            raise RuntimeError("Table path is outside its immutable publication archive.")\n    if int(manifest.get("current_predictions", 0)) <= 0:\n        raise RuntimeError("Publication has no predictions.")\n    return manifest\n', 'aews.presentation': '"""Read-only publication views and exact linear-score explanations."""\n', 'aews.presentation.explainability': '"""Exact underlying linear-score explanations using the training population as reference."""\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import Pipeline\n\n\nOUTCOMES = ("excellent", "pass", "fail", "withdrawn")\n\n\ndef fitted_linear_pipeline(model):\n    if isinstance(model, Pipeline):\n        pipeline = model\n    else:\n        calibrated = getattr(model, "calibrated_classifiers_", None)\n        if calibrated is None or len(calibrated) != 1:\n            raise ValueError("Expected the saved single four-class model, not a model ensemble.")\n        member = calibrated[0]\n        pipeline = getattr(member, "estimator", None)\n        if pipeline is None:\n            pipeline = getattr(member, "base_estimator", None)\n    if not isinstance(pipeline, Pipeline):\n        raise TypeError("The saved model does not contain the expected fitted feature pipeline.")\n    if not isinstance(pipeline.named_steps.get("classifier"), LogisticRegression):\n        raise TypeError("Exact linear-score explanations require the saved logistic classifier.")\n    if set(pipeline.classes_) != set(OUTCOMES):\n        raise ValueError("The model must predict exactly the four agreed outcomes.")\n    return pipeline\n\n\nclass OutcomeExplanations:\n    def __init__(self, model, training):\n        self.model = model\n        self.pipeline = fitted_linear_pipeline(model)\n        self.features = list(self.pipeline.feature_names_in_)\n        if training.empty or not set(self.features) <= set(training.columns):\n            raise ValueError("The reference must contain the actual training rows and model features.")\n        self.classes = list(self.pipeline.classes_)\n        self.coefficients = self.pipeline.named_steps["classifier"].coef_\n        self.intercepts = self.pipeline.named_steps["classifier"].intercept_\n        self.transformed_names = list(self.pipeline.named_steps["imputer"].get_feature_names_out(self.features))\n        self.groups = [self.features.index(name.removeprefix("missingindicator_")) for name in self.transformed_names]\n        self.training_z = self.transform(training)\n        self.reference = self.training_z.mean(axis=0)\n        self.baseline_scores = self.reference @ self.coefficients.T + self.intercepts\n\n    def transform(self, frame):\n        values = frame.loc[:, self.features].astype(float)\n        if np.isinf(values.to_numpy()).any():\n            raise ValueError("Infinite model feature encountered.")\n        result = np.asarray(self.pipeline[:-1].transform(values), dtype=float)\n        if not np.isfinite(result).all():\n            raise ValueError("The saved feature pipeline produced nonfinite values.")\n        return result\n\n    def contributions(self, frame, outcome):\n        if outcome not in self.classes:\n            raise ValueError(f"Unknown outcome: {outcome}")\n        transformed = self.transform(frame)\n        index = self.classes.index(outcome)\n        terms = (transformed - self.reference) * self.coefficients[index]\n        grouped = np.zeros((len(frame), len(self.features)))\n        for position, group in enumerate(self.groups):\n            grouped[:, group] += terms[:, position]\n        expected = self.pipeline.decision_function(frame.loc[:, self.features].astype(float))[:, index]\n        np.testing.assert_allclose(self.baseline_scores[index] + grouped.sum(axis=1), expected, atol=1e-9)\n        return pd.DataFrame(grouped, index=frame.index, columns=self.features)\n\n    def training_importance(self, training):\n        rows = []\n        for outcome in OUTCOMES:\n            values = self.contributions(training, outcome).abs().mean()\n            for feature, magnitude in values.items():\n                rows.append({"outcome": outcome, "feature": feature, "mean_absolute_score_contribution": float(magnitude)})\n        return pd.DataFrame(rows)\n\n    def result_drivers(self, current, predictions):\n        if len(current) != len(predictions):\n            raise ValueError("Current features and predictions must be aligned one-to-one.")\n        expected = self.model.predict_proba(current.loc[:, self.features].astype(float))\n        model_classes = list(self.model.classes_)\n        for outcome in OUTCOMES:\n            np.testing.assert_allclose(expected[:, model_classes.index(outcome)],\n                                       predictions["p_" + outcome].to_numpy(), rtol=1e-5, atol=1e-7)\n        predicted = np.asarray(model_classes)[expected.argmax(axis=1)]\n        if not np.array_equal(predicted, predictions.predicted_class.to_numpy()):\n            raise ValueError("The saved model does not reproduce the published outcome labels.")\n        drivers = [None] * len(current)\n        for outcome in OUTCOMES:\n            indices = np.flatnonzero(predicted == outcome)\n            if not len(indices):\n                continue\n            terms = self.contributions(current.iloc[indices], outcome).to_numpy()\n            for position, row_index in enumerate(indices):\n                top = np.argsort(np.abs(terms[position]))[::-1][:3]\n                drivers[row_index] = [\n                    {"feature": self.features[i], "score_contribution": float(terms[position, i]),\n                     "direction": "raises" if terms[position, i] > 0 else "lowers" if terms[position, i] < 0 else "neutral"}\n                    for i in top\n                ]\n        return drivers\n', 'aews.presentation.reader': '"""Read-only inspection of one complete, immutable publication."""\nimport hashlib\nimport json\n\nimport numpy as np\nimport pandas as pd\n\nfrom aews.runtime import CURRENT_MANIFEST, TABLES, validate_manifest, validate_session\nfrom aews.presentation.explainability import OutcomeExplanations\n\n\ndef read_publication(spark, config):\n    validate_session(spark, config)\n    import notebookutils\n    import mlflow.sklearn\n    fs = notebookutils.fs\n    if not fs.exists(CURRENT_MANIFEST):\n        raise RuntimeError("No current publication. Complete the full build notebook first.")\n    manifest = validate_manifest(json.loads(fs.head(CURRENT_MANIFEST, 16 * 1024 * 1024)))\n    if manifest.get("workspace_name") != config["intended_workspace"] or manifest.get("lakehouse_name") != config["intended_lakehouse"]:\n        raise RuntimeError("Manifest belongs to a different configured target.")\n    frames = {}\n    for name in TABLES:\n        frame = spark.read.format("delta").load(manifest["table_paths"][name])\n        count = frame.count()\n        if count != manifest["table_counts"][name] or count > 100000:\n            raise RuntimeError("Publication count mismatch or bounded driver read exceeded: " + name)\n        frames[name] = frame.toPandas()\n    current = frames["silver.current_weekly_features"].sort_values("enrollment_id").reset_index(drop=True)\n    predictions = frames["gold.student_course_predictions"].sort_values("enrollment_id").reset_index(drop=True)\n    if any(name in current for name in ("label", "label_status", "label_available_at")):\n        raise RuntimeError("Current publication contains future outcome fields.")\n    if predictions.prediction_id.duplicated().any() or current.enrollment_id.duplicated().any():\n        raise RuntimeError("Duplicate current keys.")\n    if len(current) != len(predictions) or not np.array_equal(current.enrollment_id, predictions.enrollment_id):\n        raise RuntimeError("Current features and predictions are not aligned.")\n    if not predictions.model_version.eq(manifest["model_version"]).all() or not predictions.mlflow_run_id.eq(manifest["mlflow_run_id"]).all():\n        raise RuntimeError("Published prediction provenance mismatch.")\n    historical = frames["silver.historical_weekly_features"].sort_values(["term", "course_id", "enrollment_id"])\n    training = historical.loc[historical.term.isin([1, 2]) & historical.label.notna() & (historical.cutoff_week == 4)]\n    # MLflow deserializes a model. Use only this trusted workspace\'s own publication.\n    model = mlflow.sklearn.load_model("runs:/" + manifest["mlflow_run_id"] + "/model")\n    explainer = OutcomeExplanations(model, training)\n    fingerprint = hashlib.sha256(pd.util.hash_pandas_object(\n        training[explainer.features + ["label", "enrollment_id"]], index=False).values.tobytes()).hexdigest()\n    if fingerprint != manifest["train_data_sha256"]:\n        raise RuntimeError("Training reference does not match the fitted model\'s provenance.")\n    drivers = explainer.result_drivers(current, predictions)\n    predictions["top_predicted_class_drivers"] = [json.dumps(value) for value in drivers]\n    if json.loads(fs.head(CURRENT_MANIFEST, 16 * 1024 * 1024)) != manifest:\n        raise RuntimeError("Publication changed during reading; rerun after the writer has finished.")\n    return {"manifest": manifest, "frames": frames, "training": training, "current": current,\n            "predictions": predictions, "explainer": explainer}\n'}
EMBEDDED_PACKAGES = ['aews', 'aews.presentation']
for name, source in EMBEDDED_MODULES.items():
    module = types.ModuleType(name)
    module.__file__ = '<embedded:' + name + '>'
    module.__package__ = name if name in EMBEDDED_PACKAGES else name.rpartition('.')[0]
    if name in EMBEDDED_PACKAGES:
        module.__path__ = []
    sys.modules[name] = module
    if '.' in name:
        parent, _, child = name.rpartition('.')
        setattr(sys.modules[parent], child, module)
    exec(compile(source, module.__file__, 'exec'), module.__dict__)
print('Self-contained Academic Early Warning System source loaded.')


In [ ]:
import json
SOURCE_CONFIG = json.loads('{"seed": 271828, "mode": "small", "profiles": {"small": {"students": 400, "courses_per_term": 8, "enrollments_per_course": 75, "assignments_per_course": 20, "sections_per_course": 3}, "full": {"students": 5000, "courses_per_term": 80, "enrollments_per_course": 150, "assignments_per_course": 20, "sections_per_course": 3}}, "terms": 5, "term_days": 84, "cutoff_weeks": [4], "current_week": 4, "intended_workspace": "Academic Early Warning System", "intended_lakehouse": "academic_early_warning", "experiment_name": "Academic Early Warning System Training", "notice": "Synthetic course outcomes only. No real Canvas ingestion or accepted model."}')


In [ ]:
from aews.presentation.reader import read_publication
VIEW = read_publication(spark, SOURCE_CONFIG)
print('Dataset profile:', VIEW['manifest']['source_profile'])
print(json.dumps(VIEW['manifest'], indent=2))
display(VIEW['frames']['gold.data_quality_summary'])
display(VIEW['training'].head(12))
print('Terms 1-2 train. Term 3 development evaluation. Term 4 previously exposed, not independent.')


In [ ]:
from aews.presentation.explainability import OUTCOMES
GLOBAL_CONTRIBUTIONS = VIEW['explainer'].training_importance(VIEW['training'])
for outcome in OUTCOMES:
    print(outcome.upper() + ': mean absolute training-centered linear-score contributions')
    display(GLOBAL_CONTRIBUTIONS.loc[GLOBAL_CONTRIBUTIONS.outcome == outcome].sort_values(
        'mean_absolute_score_contribution', ascending=False))
import pandas as pd
evaluation = VIEW['frames']['gold.model_evaluation']
selected = evaluation.loc[evaluation.selected & (evaluation['split'] == 'term3_decision') &
                          (evaluation['slice'] == 'all')]
if len(selected) != 1:
    raise ValueError('Expected exactly one selected development evaluation.')
measured = json.loads(selected.iloc[0].metrics_json)
display(pd.DataFrame([{'outcome': outcome,
    **{key: measured['classification_report'][outcome][key] for key in ('precision', 'recall', 'f1-score', 'support')},
    **{key: measured['per_class'][outcome][key] for key in ('prevalence', 'average_precision', 'brier')}}
    for outcome in OUTCOMES]))
display(pd.DataFrame(measured['confusion_matrix'], index=measured['confusion_class_order'],
                     columns=measured['confusion_class_order']))
print('Confusion matrix: actual rows, predicted columns. Full baseline and slice metrics remain in model_evaluation.')
print('Synthetic development metrics only; no independent institutional accuracy claim.')


In [ ]:
# Enrollment ID comes from canvas.enrollments.id, carried into features and predictions.
# It identifies one student's enrollment in one course, not just the student.
# predicted_class comes from the model's highest probability across the four outcomes.
# It is a prediction, NOT a known future course result.
available_predictions = VIEW['predictions'].sort_values('enrollment_id')
if available_predictions.empty:
    raise ValueError('No current predictions are available; complete publication first.')
display(available_predictions[['enrollment_id', 'course_id', 'predicted_class',
    'p_excellent', 'p_pass', 'p_fail', 'p_withdrawn', 'top_predicted_class_drivers']].head(30))

# Prefilled from real rows in THIS synthetic publication, so both small/full profiles work.
# To choose another enrollment, replace the next expression with a displayed enrollment_id.
ENROLLMENT_TO_EXPLAIN = int(available_predictions.iloc[0].enrollment_id)
selected_prediction = available_predictions.loc[
    available_predictions.enrollment_id == ENROLLMENT_TO_EXPLAIN]
if len(selected_prediction) != 1:
    raise ValueError('Choose one enrollment_id from the current predictions table.')

# Automatically prefill the selected enrollment's actual predicted outcome.
# Or replace this expression with 'excellent', 'pass', 'fail', or 'withdrawn'.
# Choosing another outcome explains THAT class's score; it does not change the prediction.
OUTCOME_TO_EXPLAIN = str(selected_prediction.iloc[0].predicted_class)
print(f'ENROLLMENT_TO_EXPLAIN = {ENROLLMENT_TO_EXPLAIN}')
print(f'OUTCOME_TO_EXPLAIN = {OUTCOME_TO_EXPLAIN!r}')
print('Run the next cell to explain these populated values. No training or prediction is changed.')


In [ ]:
row = VIEW['current'].loc[VIEW['current'].enrollment_id == ENROLLMENT_TO_EXPLAIN]
prediction = VIEW['predictions'].loc[VIEW['predictions'].enrollment_id == ENROLLMENT_TO_EXPLAIN]
if len(row) != 1 or len(prediction) != 1:
    raise ValueError('Choose one published enrollment.')
if OUTCOME_TO_EXPLAIN not in OUTCOMES:
    raise ValueError('OUTCOME_TO_EXPLAIN must be excellent, pass, fail, or withdrawn.')
outcome = OUTCOME_TO_EXPLAIN
contributions = VIEW['explainer'].contributions(row, outcome)
baseline = float(VIEW['explainer'].baseline_scores[VIEW['explainer'].classes.index(outcome)])
display(prediction)
display(contributions.T.rename(columns={row.index[0]: 'signed_score_contribution'}))
print({'outcome': outcome, 'training_reference_score': baseline,
       'contribution_sum': float(contributions.sum(axis=1).iloc[0]),
       'underlying_score': baseline + float(contributions.sum(axis=1).iloc[0])})
print('All terms reconstruct the underlying score. Top-three drivers alone do not.')
